In [ ]:
'''
Script to manage embeddings of game rules in qdrant vector store
'''

# Loading of needed packages and env variables

In [1]:

# For loading documents
from langchain_community.document_loaders import PyMuPDFLoader 
from pathlib import Path
#from langchain_community.document_loaders.parsers import TesseractBlobParser #In case there are images with text


# For chunking docs
from langchain_text_splitters import RecursiveCharacterTextSplitter 
#I decided not to use RecursiveCharacterTextSplitter. Maybe it is useful for regular documents, but not for boardgames rulebooks
#Character was also not great since the first splitting hierarchy is /n/n. Decided to keep it with recursive

# Load env vars for keys
import os
from dotenv import load_dotenv
load_dotenv()  # Loads from .env

# For embeddings model
COHERE_API_KEY=os.getenv("COHERE_TOKEN")
os.environ["COHERE_API_KEY"]=COHERE_API_KEY 
from langchain_cohere import CohereEmbeddings

# For Qdrant vector store
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams


In [2]:
# Set working directory
try:
    # Works in regular Python scripts
    base_dir = Path(__file__).resolve().parent.parent
except NameError:
    # Fallback for Jupyter notebooks and interactive shells
    base_dir = Path().resolve().parent

In [3]:
#This starts a local (embedded) Qdrant instance in the path given
path_qdrant_vector_store = base_dir / "data" /  "qdrant"

In [4]:
client = QdrantClient(path=path_qdrant_vector_store)

In [5]:
embeddings = CohereEmbeddings(model="embed-v4.0")


# Upload documents to vector store

### Set vector store

In [ ]:
#This creates a collection (like a database, a group of tables) in Qdrant. Qdrant is basically a sqlite database
# Done only if collection does not exist. Otherwise it will throw an error
client.create_collection(
    collection_name="document_embeddings",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

In [7]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name="document_embeddings",
    embedding=embeddings,
)

### Loading of documents 

In [13]:
path_rules_pdf = base_dir / "data" / 'raw-rulebooks' 

print(path_rules_pdf)

C:\Users\Carlos Ivan\Documents\Projects\boardgames-assistant\data\raw-rulebooks


In [14]:
pdf_files = [f.name for f in path_rules_pdf.glob("*.pdf")]
print(len(pdf_files))
print(pdf_files)

35
['axis-allies-rules-1942-2nd-edition.pdf', 'Axis_&_Allies_1942_2nd_Reference_v1.3.pdf', 'Grand_Austria_Hotel_game_aid_-_2_pages_v1.01.pdf', 'Grand_Austria_Hotel_Plain_and_Simple_Guide.pdf', 'Grand_Hotel_Austria_rules_EN_web.pdf', 'In_The_Footsteps_Of_Marco_Polo_-_English_Overview_Cards.pdf', 'LlamaLand_Player_Aid_(English)_version_2.pdf', 'LlamaLand_Rules_EN.pdf', 'Memoir_44_-_a_beginners_reference.pdf', 'memoir_44_rules_part1_en.pdf', 'Memoir_44_Unofficial_FAQ_v12.pdf', 'munchkindisneyrules.pdf', 'MunchkinDisney_Instr.pdf', 'MYSTERIUM_PARK_RULES_EN_BD.pdf', 'Mysterium_Park_Rules_Summary_unnofficial.pdf', 'Paleo_FAQ_11-12-20_ENG.pdf', 'Paleo_Rulebook_EN_compressed.pdf', 'pandemic season 1 rules.pdf', 'partners.pdf', 'Partners_rules.pdf', 'Power Grid Expansion Benelux Central Europe.pdf', 'Power Grid Expansion India Australia.pdf', 'Power Grid Expansion UK Ireland Northern Europe.pdf', 'Power-Grid-Recharged-Rules.pdf', 'skull_king_rulebook_optimized.pdf', 'splendor.pdf', 'Splendor_Br

In [15]:
for a_pdf in pdf_files:

    a_loader = PyMuPDFLoader(path_rules_pdf / a_pdf,
                        #mode="single", # Comment this to have different documents per page #The splitter we use appends pages, and it needs to be done without single
                        #extract_tables="markdown", #adds tables at the end of the page
                        #images_inner_format="html-img",  #Did not see huge improvements when using it 
                        #images_parser=TesseractBlobParser(),
                        )
    
    a_doc = a_loader.load()
    # It took less than 1 second without tables
    # extract_tables takes significatnly longer than without it (20 seconds)
    # with tables and images using tesseract (good quality) it took 29 seconds

    print(f'Number of characters of {a_pdf}: {sum([len(a.page_content) for a in a_doc])}') #Total characters

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,  # chunk size (characters)
        chunk_overlap=300,  # chunk overlap (characters)
        add_start_index=True,  # track index in original document
    )

    all_splits = text_splitter.split_documents(a_doc)
    print(f"Split blog post into {len(all_splits)} sub-documents.")

    # Append to each split name of pdf (in case metadata is not included in )
    for a_split in all_splits:
        # added to the chunks both the name of the rule book and the page from the chunk comes from (useful to ask in which page to find something) 
        a_split.page_content = 'Game: ' + a_pdf.split('.')[0] + f', page {a_split.metadata["page"]}' + '. Content: ' + a_split.page_content
        a_split.metadata['game']= a_pdf.split('.')[0] 

    # Add embeddings of the splits into the vector store
    _ = vector_store.add_documents(documents=all_splits)
    #print(f'Size of vector store until now: {len(vector_store.store.items())}') #Works for in-memory vector_store
    nr_vector_store_documents=client.count(collection_name=vector_store.collection_name).count  # Works for Qdrant vector_store
    print(f'Size of vector store until now: {nr_vector_store_documents}\n') # Works for Qdrant vector_store

Number of characters of axis-allies-rules-1942-2nd-edition.pdf: 88784
Split blog post into 81 sub-documents.
Size of vector store until now: 81

Number of characters of Axis_&_Allies_1942_2nd_Reference_v1.3.pdf: 6494
Split blog post into 6 sub-documents.
Size of vector store until now: 87

Number of characters of Grand_Austria_Hotel_game_aid_-_2_pages_v1.01.pdf: 8765
Split blog post into 8 sub-documents.
Size of vector store until now: 95

Number of characters of Grand_Austria_Hotel_Plain_and_Simple_Guide.pdf: 24438
Split blog post into 22 sub-documents.
Size of vector store until now: 117

Number of characters of Grand_Hotel_Austria_rules_EN_web.pdf: 32944
Split blog post into 31 sub-documents.
Size of vector store until now: 148

Number of characters of In_The_Footsteps_Of_Marco_Polo_-_English_Overview_Cards.pdf: 4928
Split blog post into 5 sub-documents.
Size of vector store until now: 153

Number of characters of LlamaLand_Player_Aid_(English)_version_2.pdf: 0
Split blog post into 

Retrying langchain_cohere.embeddings.CohereEmbeddings.embed_with_retry.<locals>._embed_with_retry in 4.0 seconds as it raised TooManyRequestsError: status_code: 429, body: {'id': '552c81a6-b7a0-429c-8c0b-cee41b524b50', 'message': 'trial token rate limit exceeded, limit is 100000 tokens per minute'}.


Size of vector store until now: 550

Number of characters of splendor.pdf: 9683
Split blog post into 9 sub-documents.
Size of vector store until now: 559

Number of characters of Splendor_Brief_by_Liumas_2014-05b.pdf: 3053
Split blog post into 4 sub-documents.
Size of vector store until now: 563

Number of characters of Take-5-or-6-Nimmt-Rules.pdf: 14888
Split blog post into 16 sub-documents.
Size of vector store until now: 579

Number of characters of Take_5_6nimmt30_rules_english.pdf: 28191
Split blog post into 28 sub-documents.
Size of vector store until now: 607

Number of characters of The_Crew_Demo_Sheet.pdf: 3207
Split blog post into 3 sub-documents.
Size of vector store until now: 610

Number of characters of The_Crew_Manual.pdf: 33558
Split blog post into 35 sub-documents.
Size of vector store until now: 645

Number of characters of Tureluurs official rules.pdf: 11346
Split blog post into 11 sub-documents.
Size of vector store until now: 656

Number of characters of Tureluurs 

# Qdrant database management

You can access the contents of the Qdrant collection as if it was an sqlite database, because that is what it is

## Using sqlite3

For this, you have to create a connection to the database (and dont forget to close it once you are done)

### Start connection

In [6]:
import sqlite3

# Path to your SQLite file
db_path = path_qdrant_vector_store / 'collection' / 'document_embeddings' / 'storage.sqlite'


In [7]:
conn = sqlite3.connect(db_path)


In [8]:
cursor = conn.cursor()

### See all tables in the collection

In [9]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables:", [table[0] for table in tables])


Tables: ['points']


### See tables schemas

In [10]:
for table_name, in tables:
    print(f"\nSchema of {table_name}:")
    cursor.execute(f"PRAGMA table_info({table_name});")
    schema = cursor.fetchall()
    for col in schema:
        # Each col: (cid, name, type, notnull, dflt_value, pk)
        print(f"  {col[1]} ({col[2]}) {'NOT NULL' if col[3] else ''} {'PRIMARY KEY' if col[5] else ''}")



Schema of points:
  id (TEXT)  PRIMARY KEY
  point (BLOB)  


### See number of rows

In [11]:
for table_name, in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table_name};")
    count = cursor.fetchone()[0]
    print(f"Number of rows in {table_name}: {count}")


Number of rows in points: 308


### Inspect a record

In [12]:
cursor.execute(f"SELECT * FROM points LIMIT 1;")
row = cursor.fetchone()
print(row)

('gASVJAAAAAAAAACMIDVkZjkzMTZiNGExMDRhZGE4MWRkODZiMDk3ODM4Y2VklC4=', b'\x80\x04\x95S<\x00\x00\x00\x00\x00\x00\x8c qdrant_client.http.models.models\x94\x8c\x0bPointStruct\x94\x93\x94)\x81\x94}\x94(\x8c\x08__dict__\x94}\x94(\x8c\x02id\x94\x8c 5df9316b4a104ada81dd86b097838ced\x94\x8c\x06vector\x94}\x94\x8c\x00\x94]\x94(G?\x97\xc0\x00\n\xbc\xc7qG\xbf\x91\x9f\xff\xfe\xed\x1fAG?\xad\x80\x00\x04K\x82\xfaG?\x91?\xff\xfd\xda>\x83G\xbf\x9b\x80\x00\x04K\x82\xfaG\xbf\x9f?\xff\xfd\xda>\x83G?\x9d?\xff\xfd\xda>\x83G\xbf\x94?\xff\xf5C8\x8fG?\x9a\x1f\xff\xfa\xa1\x9cGG\xbf\x95\x80\x00\x04K\x82\xfaG\xbf\x84\xc0\x00\x02%\xc1}G\xbf\x9e\xc0\x00\x02%\xc1}G\xbf\xa5?\xff\xfd\xda>\x83G\xbfi\xbf\xff\xf9\x8e\xbb\x89G\xbfT?\xff\xfc"p\x85G?\x8d@\x00\x0f\x08JkG?\x93\x9f\xff\xfe\xed\x1fAG?\x9e\xc0\x00\x02%\xc1}G\xbf\x87\xc0\x00\n\xbc\xc7qG?}`\x00\t\xa9\xe6\xb3G\xbf\xa8\x9f\xff\xfe\xed\x1fAG?\x87?\xff\xfd\xda>\x83G?\x80\x1f\xff\xfa\xa1\x9cGG\xbf\x81@\x00\x0f\x08JkG\xbf\x80\x00\x00\x00\x00\x00\x00G?\xad?\xff\xfd\xda>\x

In [13]:
row[1]

b'\x80\x04\x95S<\x00\x00\x00\x00\x00\x00\x8c qdrant_client.http.models.models\x94\x8c\x0bPointStruct\x94\x93\x94)\x81\x94}\x94(\x8c\x08__dict__\x94}\x94(\x8c\x02id\x94\x8c 5df9316b4a104ada81dd86b097838ced\x94\x8c\x06vector\x94}\x94\x8c\x00\x94]\x94(G?\x97\xc0\x00\n\xbc\xc7qG\xbf\x91\x9f\xff\xfe\xed\x1fAG?\xad\x80\x00\x04K\x82\xfaG?\x91?\xff\xfd\xda>\x83G\xbf\x9b\x80\x00\x04K\x82\xfaG\xbf\x9f?\xff\xfd\xda>\x83G?\x9d?\xff\xfd\xda>\x83G\xbf\x94?\xff\xf5C8\x8fG?\x9a\x1f\xff\xfa\xa1\x9cGG\xbf\x95\x80\x00\x04K\x82\xfaG\xbf\x84\xc0\x00\x02%\xc1}G\xbf\x9e\xc0\x00\x02%\xc1}G\xbf\xa5?\xff\xfd\xda>\x83G\xbfi\xbf\xff\xf9\x8e\xbb\x89G\xbfT?\xff\xfc"p\x85G?\x8d@\x00\x0f\x08JkG?\x93\x9f\xff\xfe\xed\x1fAG?\x9e\xc0\x00\x02%\xc1}G\xbf\x87\xc0\x00\n\xbc\xc7qG?}`\x00\t\xa9\xe6\xb3G\xbf\xa8\x9f\xff\xfe\xed\x1fAG?\x87?\xff\xfd\xda>\x83G?\x80\x1f\xff\xfa\xa1\x9cGG\xbf\x81@\x00\x0f\x08JkG\xbf\x80\x00\x00\x00\x00\x00\x00G?\xad?\xff\xfd\xda>\x83G\xbf\x9c\x00\x00\x00\x00\x00\x00G?\x9b\xc0\x00\n\xbc\xc7qG?\xa3\x0

### Close connection

In [14]:
conn.close()


## Using Qdrant

In [6]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name="document_embeddings",
    embedding=embeddings,
)

### Counting number of records

In [16]:
#client.count(collection_name=vector_store.collection_name).count
client.count(collection_name= 'document_embeddings' ).count

721

### Deleting all records

In [8]:
# Access the underlying Qdrant client
client = vector_store.client
collection_name = vector_store.collection_name

# Get all points in the collection
all_points = client.scroll(collection_name=collection_name, limit=client.count(collection_name= collection_name ).count)[0]
all_ids = [point.id for point in all_points]


In [9]:
len(all_ids)

364

In [10]:
vector_store.delete(ids=all_ids)


True

In [11]:
client.count(collection_name= collection_name ).count

0